In [11]:
from pathlib import Path

import warnings

import polars as pl

warnings.filterwarnings("ignore")

pl.Config.set_tbl_rows(10)
pl.Config.set_tbl_cols(20)

polars.config.Config

In [12]:
PROJECT_ROOT = Path.cwd().parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"

INTERIM_DIR = PROJECT_ROOT / "data" / "interim"

INTERIM_DIR.mkdir(parents=True, exist_ok=True)

In [13]:
products_df = pl.read_csv(RAW_DIR / "Products.csv")

warehouses_df = pl.read_csv(RAW_DIR / "Warehouses.csv")

sales_df = pl.read_csv(RAW_DIR / "Sales_History.csv")

inventory_df = pl.read_csv(RAW_DIR / "Inventory.csv")

In [14]:
required_sales_columns = [
    "product_id",
    "hub_id",
    "year",
    "month",
]

missing = [
    col
    for col in required_sales_columns
    if col not in sales_df.columns
]

assert len(missing) == 0, f"Missing columns: {missing}"

print("Sales schema validated.")

Sales schema validated.


In [15]:
sales_df = sales_df.with_columns(
    [
        pl.col("year").cast(pl.Int16),
        pl.col("month").cast(pl.Int8),
        pl.col("promotion").cast(pl.Int8),
        pl.col("festival_flag").cast(pl.Int8),
    ]
)

In [16]:
merged_df = (
    sales_df
    .join(
        products_df,
        on="product_id",
        how="left",
    )
    .join(
        warehouses_df,
        on="hub_id",
        how="left",
    )
    .join(
        inventory_df,
        on=["hub_id", "product_id"],
        how="left",
    )
)

In [17]:
print(merged_df.shape)

display(merged_df.head())

display(merged_df.null_count())

(180000, 19)


year,month,hub_id,product_id,opening_stock,quantity_sold,promotion,discount_percentage,festival_flag,returns,closing_stock,product_name,sku,category,brand,hub_name,city,current_stock,last_updated
i16,i8,str,str,i64,i64,i8,i64,i8,i64,i64,str,str,str,str,str,str,i64,str
2023,1,"""HUB001""","""P000001""",527,132,0,0,1,6,401,"""Apple Fashion 1""","""SKU00001""","""Fashion""","""Apple""","""Chennai Hub""","""Chennai""",216,"""2025-12-31"""
2023,1,"""HUB001""","""P000002""",506,50,0,0,1,0,456,"""LG Sports 2""","""SKU00002""","""Sports""","""LG""","""Chennai Hub""","""Chennai""",808,"""2025-12-31"""
2023,1,"""HUB001""","""P000003""",427,111,0,0,1,2,318,"""Samsung Home 3""","""SKU00003""","""Home""","""Samsung""","""Chennai Hub""","""Chennai""",418,"""2025-12-31"""
2023,1,"""HUB001""","""P000004""",531,77,0,0,1,3,457,"""Puma Fashion 4""","""SKU00004""","""Fashion""","""Puma""","""Chennai Hub""","""Chennai""",431,"""2025-12-31"""
2023,1,"""HUB001""","""P000005""",855,28,0,0,1,1,828,"""Apple Electronics 5""","""SKU00005""","""Electronics""","""Apple""","""Chennai Hub""","""Chennai""",205,"""2025-12-31"""


year,month,hub_id,product_id,opening_stock,quantity_sold,promotion,discount_percentage,festival_flag,returns,closing_stock,product_name,sku,category,brand,hub_name,city,current_stock,last_updated
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [18]:
merged_df.write_parquet(
    INTERIM_DIR / "merged.parquet"
)

print("Merged dataset saved successfully.")

Merged dataset saved successfully.
